<a href="https://colab.research.google.com/github/AdamPrzychodniPrivate/somali-radios-with-ai-for-food-security/blob/main/1_phase/Radio_Ergo_Somali_Speech_to_Text_for_Food_Security.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📻 Radio Ergo Somali Speech-to-Text for Food Security

## 🎯 Project Overview

**Radio Ergo SoundCloud Audio Downloader and Transcriber**

This notebook automates the end-to-end process of downloading Somali-language audio from **Radio Ergo’s SoundCloud channel** and transcribing it into readable text. It also includes a comprehensive evaluation of three different **speech-to-text models** to determine the most reliable solution for Somali transcription in support of food security analysis.

📘 **Full Project Report:**  
  - 📄 [Leveraging Local Radio for Real-Time Food Security Insights: An AI-Powered Approach](https://docs.google.com/document/d/1CvSWsIIZN1jriA02DhczZwuu_-vq87tMO50ZeyycRzY/edit?usp=sharing)

---

## 1. 📥 Data Collection – Audio Download System

The notebook includes a fully automated system for acquiring audio data from Radio Ergo:

### ✅ Features:

- **🔗 URL Validation:**  
  Ensures SoundCloud links are properly formatted.

- **📆 Date-Based Extraction:**  
  Extracts track links from Radio Ergo’s profile within a specified date range.

- **📡 Automated Download:**  
  Downloads all matching audio files in **MP3** format.

- **💾 Storage Options:**  
  Saves audio locally or to **Google Drive** for persistent access and processing.

---

## 2. 🧠 Transcription Model Evaluation

The notebook tests **three speech-to-text models** to find the most accurate transcription system for Somali-language audio.

### 🧪 Model 1: OpenAI Whisper (Standard)

- **🔍 Description:** A general-purpose transcription model by OpenAI.
- **✅ Language Detection:** Recognizes Somali.
- **❌ Major Issue:** Outputs **Arabic script** instead of the **Somali Latin script**.
- **📉 Result:** Unusable transcription with repeated nonsensical Arabic words.

---

### 🧪 Model 2: Whisper Small Somali (`steja/whisper-small-somali`)

- **🔍 Description:** A specialized model from Hugging Face fine-tuned for Somali.
- **✅ Improvement:** Correctly uses **Somali Latin script**.
- **❌ Major Issue:** Exhibits **hallucinatory repetition**, e.g., *"iyo iyo iyo"*, *"dhul dhul dhul"*.
- **📉 Result:** Partially successful, but unreliable due to extreme repetition.

---

### 🧪 Model 3: Gemini 2.0 Flash (Google)

- **🔍 Description:** A high-performance transcription model from Google.
- **✅ Improvement:** Produces **accurate**, **coherent**, and **well-structured** text.
- **✅ Script:** Correct **Somali Latin script**.
- **✅ Repetition:** Minimal and natural.
- **📈 Result:** **Success.** Best-performing model for Somali transcription.

---

## 3. 📊 Transcription Analysis

The final section of the notebook performs a detailed evaluation of each model’s output using quantitative metrics.

### 🧮 Metrics Analyzed:

- **🔠 Word Frequencies:** Frequency of unique and repeated words.
- **🔁 Repetition Patterns:** Identification of hallucinated or natural repetitions.
- **📏 Line Length Statistics:** Structural consistency of transcription lines.

### ✅ Outcome:
- **Model 1 & 2:** Failed to deliver reliable transcriptions.
- **Model 3 (Gemini 2.0 Flash):** Achieved **high accuracy** and **production-quality output**.

---

## ✅ Final Recommendation

For any future transcription of Somali-language audio — especially in humanitarian or food security contexts — **Gemini 2.0 Flash** is currently the most effective and reliable solution. It significantly outperforms general and specialized Whisper models in both accuracy and formatting.



#### 📋 INSTRUCTION

## ⚡️ How to Speed Up Speech-to-Text Models:

1. 🖱️ Click **Resources** in the menu
2. 🔄 Select **Change runtime type**
3. 🖥️ Choose a **GPU-enabled runtime** 🚀

> ℹ️ **Note**: Following these steps will significantly improve processing speed for speech recognition tasks.

# 1. Data Collection - Audio Download System

Code below automates downloading audio from SoundCloud by validating URLs, extracting track links from a profile page within a specified date range, and downloading the audio files using `yt-dlp`.

It first scrapes a given SoundCloud profile for track links, filters them based on date patterns in the URLs, and downloads matching audio tracks in MP3 format.

Additionally, it includes an option to save downloaded files to Google Drive for storage.


In [1]:
"""
SoundCloud Audio Downloader

A clean, efficient tool for downloading SoundCloud tracks by date range or specific URLs.
Designed for Linux environments with proper error handling and logging.
"""

import os
import re
import time
import logging
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Optional, Union
from urllib.parse import urljoin

import yt_dlp
import requests
from bs4 import BeautifulSoup


# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('soundcloud_downloader.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


class SoundCloudDownloader:
    """
    A class for downloading SoundCloud tracks with date filtering capabilities.
    
    Handles URL validation, audio extraction, and batch downloading operations
    with proper error handling and logging.
    """
    
    # Month name mappings for date parsing
    MONTH_NAMES = {
        'january': 1, 'february': 2, 'march': 3, 'april': 4, 'may': 5, 'june': 6,
        'july': 7, 'august': 8, 'september': 9, 'october': 10, 'november': 11, 'december': 12,
        'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'jun': 6, 'jul': 7,
        'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
    }
    
    # SoundCloud URL validation pattern
    SOUNDCLOUD_URL_PATTERN = r'^https?://(?:www\.)?soundcloud\.com/[\w-]+/[\w-]+'
    
    # Date extraction patterns from URLs
    DATE_PATTERNS = [
        r'(\d{1,2})-([a-z]+)-(\d{4})',  # DD-month-YYYY or DD-mon-YYYY
        r'(\d{4})-(\d{1,2})-(\d{1,2})'  # YYYY-MM-DD
    ]
    
    def __init__(self, output_base_dir: Optional[str] = None, audio_quality: str = "192"):
        """
        Initialize the SoundCloud downloader.
        
        Args:
            output_base_dir: Base directory for downloads (default: project data directory)
            audio_quality: Audio quality for MP3 conversion (default: 192kbps)
        """
        if output_base_dir is None:
            # Default to project structure relative to notebook location
            notebook_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
            project_root = self._find_project_root(notebook_dir)
            self.output_base_dir = project_root / "data" / "01_raw" / "comparison"
        else:
            self.output_base_dir = Path(output_base_dir)
            
        self.audio_quality = audio_quality
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
        
    def _find_project_root(self, start_path: Path) -> Path:
        """
        Find the project root directory by looking for the project folder.
        
        Args:
            start_path: Starting directory to search from
            
        Returns:
            Path to project root directory
        """
        current = start_path.resolve()
        
        # Look for the project directory name in the path
        project_name = "somali-radios-with-ai-for-food-security"
        
        while current != current.parent:
            if current.name == project_name:
                return current
            # Also check if project directory exists as a subdirectory
            if (current / project_name).exists():
                return current / project_name
            current = current.parent
        
        # If not found, create it in the current working directory
        project_root = Path.cwd() / project_name
        logger.info(f"Project root not found, using: {project_root}")
        return project_root
        
    def validate_soundcloud_url(self, url: str) -> bool:
        """
        Validate if the provided URL is a valid SoundCloud URL.
        
        Args:
            url: URL to validate
            
        Returns:
            True if valid SoundCloud URL, False otherwise
        """
        return bool(re.match(self.SOUNDCLOUD_URL_PATTERN, url, re.IGNORECASE))
    
    def create_output_directory(self, dir_name: str) -> Path:
        """
        Create output directory if it doesn't exist.
        
        Args:
            dir_name: Directory name to create
            
        Returns:
            Path object for the created directory
        """
        output_dir = self.output_base_dir / dir_name
        output_dir.mkdir(parents=True, exist_ok=True)
        return output_dir
    
    def get_yt_dlp_options(self, output_dir: Path) -> dict:
        """
        Get yt-dlp configuration options.
        
        Args:
            output_dir: Output directory path
            
        Returns:
            Dictionary of yt-dlp options
        """
        return {
            'format': 'bestaudio/best',
            'outtmpl': str(output_dir / '%(title)s.%(ext)s'),
            'noplaylist': True,
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': self.audio_quality,
            }],
            'quiet': True,
            'no_warnings': True
        }
    
    def download_audio(self, url: str, output_dir: Path) -> Optional[str]:
        """
        Download audio from a single SoundCloud URL.
        
        Args:
            url: SoundCloud URL to download
            output_dir: Directory to save the audio file
            
        Returns:
            Path to downloaded file if successful, None otherwise
        """
        if not self.validate_soundcloud_url(url):
            logger.error(f"Invalid SoundCloud URL: {url}")
            return None
            
        ydl_opts = self.get_yt_dlp_options(output_dir)
        
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                filename = ydl.prepare_filename(info)
                base, _ = os.path.splitext(filename)
                mp3_file = f"{base}.mp3"
                
                logger.info(f"Successfully downloaded: {mp3_file}")
                return mp3_file
                
        except Exception as e:
            logger.error(f"Error downloading {url}: {str(e)}")
            return None
    
    def extract_date_from_url(self, url: str) -> Optional[datetime]:
        """
        Extract date from SoundCloud URL using various patterns.
        
        Args:
            url: SoundCloud URL to parse
            
        Returns:
            datetime object if date found, None otherwise
        """
        url_path = url.split('/')[-1].lower()
        
        for pattern in self.DATE_PATTERNS:
            match = re.search(pattern, url_path, re.IGNORECASE)
            if match:
                try:
                    if len(match.groups()) == 3:
                        day, month_str, year = match.groups()
                        
                        # Try to convert month string to number
                        if month_str.isdigit():
                            month = int(month_str)
                        else:
                            month = self.MONTH_NAMES.get(month_str.lower())
                            
                        if month and 1 <= month <= 12:
                            return datetime(int(year), month, int(day))
                            
                except (ValueError, TypeError) as e:
                    logger.debug(f"Could not parse date from {url}: {e}")
                    continue
                    
        return None
    
    def fetch_profile_tracks(self, profile_url: str) -> List[str]:
        """
        Fetch all track URLs from a SoundCloud profile page.
        
        Args:
            profile_url: SoundCloud profile URL
            
        Returns:
            List of track URLs found on the profile
        """
        try:
            response = self.session.get(profile_url, timeout=30)
            response.raise_for_status()
        except Exception as e:
            logger.error(f"Error fetching profile page {profile_url}: {e}")
            return []
        
        soup = BeautifulSoup(response.text, 'html.parser')
        track_urls = []
        
        # Extract username from profile URL
        username = profile_url.rstrip('/').split('/')[-1]
        
        # Find all track links
        for link in soup.find_all('a', href=True):
            href = link['href']
            if (href.startswith('/') and 
                username in href and 
                '/sets/' not in href and  # Exclude playlists
                href.count('/') >= 2):    # Ensure it's a track URL format
                
                full_url = urljoin('https://soundcloud.com', href)
                if self.validate_soundcloud_url(full_url):
                    track_urls.append(full_url)
        
        # Remove duplicates while preserving order
        return list(dict.fromkeys(track_urls))
    
    def generate_potential_urls(self, profile_url: str, start_date: datetime, end_date: datetime) -> List[str]:
        """
        Generate potential URLs based on date patterns for a profile.
        
        Args:
            profile_url: Base SoundCloud profile URL
            start_date: Start date for URL generation
            end_date: End date for URL generation
            
        Returns:
            List of potential URLs to check
        """
        username = profile_url.rstrip('/').split('/')[-1]
        base_url = f"https://soundcloud.com/{username}"
        potential_urls = []
        
        current_date = start_date
        while current_date <= end_date:
            day = current_date.day
            month_full = current_date.strftime('%B').lower()
            month_abbr = current_date.strftime('%b').lower()
            year = current_date.year
            
            # Generate various URL patterns commonly used
            patterns = [
                f"idaacadda-{day:02d}-{month_abbr}-{year}",
                f"idaacadda-{day}-{month_abbr}-{year}",
                f"idaacadda-{day:02d}-{month_full}-{year}",
                f"idaacadda-{day}-{month_full}-{year}",
                f"show-{day:02d}-{month_abbr}-{year}",
                f"broadcast-{day}-{month_abbr}-{year}"
            ]
            
            for pattern in patterns:
                potential_urls.append(f"{base_url}/{pattern}")
                
            current_date += timedelta(days=1)
        
        return potential_urls
    
    def check_url_exists(self, url: str) -> bool:
        """
        Check if a URL exists by sending a HEAD request.
        
        Args:
            url: URL to check
            
        Returns:
            True if URL exists, False otherwise
        """
        try:
            response = self.session.head(url, timeout=10)
            return response.status_code == 200
        except:
            return False
    
    def get_urls_by_date_range(self, profile_url: str, start_date: datetime, end_date: datetime) -> List[str]:
        """
        Get SoundCloud URLs within a specific date range.
        
        Args:
            profile_url: SoundCloud profile URL
            start_date: Start date for filtering
            end_date: End date for filtering
            
        Returns:
            List of URLs within the specified date range
        """
        logger.info(f"Searching for tracks from {start_date.date()} to {end_date.date()}")
        
        # Method 1: Parse existing tracks from profile page
        profile_tracks = self.fetch_profile_tracks(profile_url)
        urls_in_range = []
        
        for url in profile_tracks:
            track_date = self.extract_date_from_url(url)
            if track_date and start_date <= track_date <= end_date:
                urls_in_range.append(url)
                logger.info(f"Found matching URL: {url}")
        
        # Method 2: Generate and check potential URLs
        logger.info("Checking for additional URLs using date patterns...")
        potential_urls = self.generate_potential_urls(profile_url, start_date, end_date)
        
        for url in potential_urls:
            if url not in urls_in_range and self.check_url_exists(url):
                urls_in_range.append(url)
                logger.info(f"Found additional URL: {url}")
                time.sleep(0.5)  # Rate limiting
        
        return urls_in_range
    
    def download_by_date_range(self, profile_url: str, start_date_str: str, end_date_str: str, 
                             output_dir_name: Optional[str] = None) -> List[str]:
        """
        Download SoundCloud tracks within a specified date range.
        
        Args:
            profile_url: SoundCloud profile URL
            start_date_str: Start date in 'YYYY-MM-DD' format
            end_date_str: End date in 'YYYY-MM-DD' format
            output_dir_name: Custom output directory name
            
        Returns:
            List of successfully downloaded file paths
        """
        try:
            start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
            end_date = datetime.strptime(end_date_str, '%Y-%m-%d')
        except ValueError as e:
            logger.error(f"Invalid date format. Use YYYY-MM-DD: {e}")
            return []
        
        if not output_dir_name:
            output_dir_name = f"soundcloud_{start_date_str}_to_{end_date_str}"
        
        output_dir = self.create_output_directory(output_dir_name)
        urls = self.get_urls_by_date_range(profile_url, start_date, end_date)
        
        if not urls:
            logger.warning("No tracks found in the specified date range")
            return []
        
        return self._download_urls(urls, output_dir)
    
    def download_specific_urls(self, urls: List[str], output_dir_name: Optional[str] = None) -> List[str]:
        """
        Download specific SoundCloud URLs.
        
        Args:
            urls: List of SoundCloud URLs to download
            output_dir_name: Custom output directory name
            
        Returns:
            List of successfully downloaded file paths
        """
        if not output_dir_name:
            output_dir_name = f"soundcloud_downloads_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
        
        output_dir = self.create_output_directory(output_dir_name)
        return self._download_urls(urls, output_dir)
    
    def _download_urls(self, urls: List[str], output_dir: Path) -> List[str]:
        """
        Internal method to download a list of URLs.
        
        Args:
            urls: List of URLs to download
            output_dir: Output directory path
            
        Returns:
            List of successfully downloaded file paths
        """
        logger.info(f"Starting download of {len(urls)} tracks to: {output_dir}")
        
        downloaded_files = []
        failed_downloads = []
        
        for i, url in enumerate(urls, 1):
            logger.info(f"Downloading {i}/{len(urls)}: {url}")
            
            result = self.download_audio(url, output_dir)
            if result:
                downloaded_files.append(result)
            else:
                failed_downloads.append(url)
            
            # Rate limiting to be respectful to SoundCloud
            time.sleep(1)
        
        # Log summary
        logger.info(f"\nDownload Summary:")
        logger.info(f"Total URLs processed: {len(urls)}")
        logger.info(f"Successfully downloaded: {len(downloaded_files)}")
        logger.info(f"Failed downloads: {len(failed_downloads)}")
        
        if failed_downloads:
            logger.warning("Failed URLs:")
            for url in failed_downloads:
                logger.warning(f"  - {url}")
        
        if downloaded_files:
            logger.info("Successfully downloaded files:")
            for file_path in downloaded_files:
                logger.info(f"  - {file_path}")
        
        return downloaded_files


# Usage examples and convenience functions
def download_radio_ergo_by_date(start_date: str, end_date: str, output_dir: Optional[str] = None) -> List[str]:
    """
    Convenience function to download Radio Ergo tracks by date range.
    Uses the default project directory structure.
    
    Args:
        start_date: Start date in 'YYYY-MM-DD' format
        end_date: End date in 'YYYY-MM-DD' format
        output_dir: Optional custom output directory name (will be created in default path)
        
    Returns:
        List of downloaded file paths
    """
    downloader = SoundCloudDownloader()
    profile_url = "https://soundcloud.com/radio-ergo"
    return downloader.download_by_date_range(profile_url, start_date, end_date, output_dir)


def download_by_date_range(profile_url: str, start_date: str, end_date: str, 
                          output_dir: Optional[str] = None) -> List[str]:
    """
    Main convenience function matching the original API for backward compatibility.
    Downloads tracks by date range using the project directory structure.
    
    Args:
        profile_url: SoundCloud profile URL
        start_date: Start date in 'YYYY-MM-DD' format  
        end_date: End date in 'YYYY-MM-DD' format
        output_dir: Optional custom output directory name
        
    Returns:
        List of downloaded file paths
    """
    downloader = SoundCloudDownloader()
    return downloader.download_by_date_range(profile_url, start_date, end_date, output_dir)


def download_urls(urls: List[str], output_dir: Optional[str] = None) -> List[str]:
    """
    Convenience function to download specific URLs.
    
    Args:
        urls: List of SoundCloud URLs
        output_dir: Optional custom output directory name
        
    Returns:
        List of downloaded file paths
    """
    downloader = SoundCloudDownloader()
    return downloader.download_specific_urls(urls, output_dir)


#### 📋 INSTRUCTION
**Run the cell below to download Radio Ergo auditions from SoundCloud within a specific date range and optionally save them to Google Drive.**

In [2]:
# Example usage:
if __name__ == "__main__":
    # Example matching your original usage pattern
    profile_url = "https://soundcloud.com/radio-ergo"
    start_date = "2025-03-15" 
    end_date = "2025-03-16"
    
    # This will save to: somali-radios-with-ai-for-food-security/data/01_raw/comparison/
    downloaded_files = download_by_date_range(profile_url, start_date, end_date)
    
    print(f"Downloaded {len(downloaded_files)} files:")
    for file_path in downloaded_files:
        print(f"  - {file_path}")
    
    # Alternative examples:
    # Example 1: Download Radio Ergo tracks for March 2025  
    # downloaded = download_radio_ergo_by_date("2025-03-01", "2025-03-31")
    
    # Example 2: Download specific URLs
    # specific_urls = [
    #     "https://soundcloud.com/radio-ergo/idaacadda-09-mar-2025",
    #     "https://soundcloud.com/radio-ergo/idaacadda-10-mar-2025"
    # ]
    # downloaded = download_urls(specific_urls, "radio_ergo_march_selection")
    
    # Example 3: Use the class directly for more control
    # downloader = SoundCloudDownloader(audio_quality="320")
    # downloaded = downloader.download_by_date_range(
    #     "https://soundcloud.com/radio-ergo", 
    #     "2025-03-09", 
    #     "2025-03-09", 
    #     "radio_ergo_march_9"
    # )

2025-09-22 12:30:56,219 - INFO - Searching for tracks from 2025-03-15 to 2025-03-16
2025-09-22 12:30:56,712 - INFO - Checking for additional URLs using date patterns...
2025-09-22 12:30:56,895 - INFO - Found additional URL: https://soundcloud.com/radio-ergo/idaacadda-15-mar-2025
2025-09-22 12:30:57,528 - INFO - Found additional URL: https://soundcloud.com/radio-ergo/idaacadda-15-march-2025
2025-09-22 12:30:58,166 - INFO - Found additional URL: https://soundcloud.com/radio-ergo/show-15-mar-2025
2025-09-22 12:30:58,839 - INFO - Found additional URL: https://soundcloud.com/radio-ergo/broadcast-15-mar-2025
2025-09-22 12:30:59,529 - INFO - Found additional URL: https://soundcloud.com/radio-ergo/idaacadda-16-mar-2025
2025-09-22 12:31:00,166 - INFO - Found additional URL: https://soundcloud.com/radio-ergo/idaacadda-16-march-2025
2025-09-22 12:31:00,811 - INFO - Found additional URL: https://soundcloud.com/radio-ergo/show-16-mar-2025
2025-09-22 12:31:01,447 - INFO - Found additional URL: https

2025-09-22 12:32:33,995 - INFO - Successfully downloaded: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/comparison/soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 15-MAR-2025.mp3
2025-09-22 12:32:34,999 - INFO - Downloading 2/8: https://soundcloud.com/radio-ergo/idaacadda-15-march-2025
ERROR: [soundcloud] Unable to download JSON metadata: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
2025-09-22 12:32:35,347 - ERROR - Error downloading https://soundcloud.com/radio-ergo/idaacadda-15-march-2025: ERROR: [soundcloud] Unable to download JSON metadata: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
2025-09-22 12:32:36,357 - INFO - Downloading 3/8: https://soundcloud.com/radio-ergo/show-15-mar-2025
ERROR: [soundcloud] Unable to download JSON metadata: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
2025-09-22 12:32:36,623 - ERROR - Error downloading https://soundcloud.com/radio-ergo/show-15-mar-2025: ERROR

2025-09-22 12:34:01,965 - INFO - Successfully downloaded: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/comparison/soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 16-MAR-2025.mp3
2025-09-22 12:34:02,969 - INFO - Downloading 6/8: https://soundcloud.com/radio-ergo/idaacadda-16-march-2025
ERROR: [soundcloud] Unable to download JSON metadata: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
2025-09-22 12:34:03,245 - ERROR - Error downloading https://soundcloud.com/radio-ergo/idaacadda-16-march-2025: ERROR: [soundcloud] Unable to download JSON metadata: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
2025-09-22 12:34:04,249 - INFO - Downloading 7/8: https://soundcloud.com/radio-ergo/show-16-mar-2025
ERROR: [soundcloud] Unable to download JSON metadata: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
2025-09-22 12:34:04,461 - ERROR - Error downloading https://soundcloud.com/radio-ergo/show-16-mar-2025: ERROR

Downloaded 2 files:
  - /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/comparison/soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 15-MAR-2025.mp3
  - /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/comparison/soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 16-MAR-2025.mp3


# 2. Transcription with Whisper Models

## Regular OpenAI Whisper Model

In [3]:
"""
Audio Transcription Tool using OpenAI Whisper

A clean, efficient tool for transcribing audio files using Whisper ASR.
Designed for Linux environments with proper error handling and logging.
"""

import os
import glob
import shutil
import subprocess
import logging
from datetime import datetime
from pathlib import Path
from typing import List, Optional, Union

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('audio_transcription.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


class AudioTranscriber:
    """
    A class for transcribing audio files using OpenAI Whisper.
    
    Handles various audio formats, automatic format conversion, and batch processing
    with proper error handling and logging.
    """
    
    # Supported audio formats
    SUPPORTED_FORMATS = ["wav", "mp3", "m4a", "flac", "ogg", "webm", "mp4"]
    
    # Whisper model sizes (from smallest to largest)
    MODEL_SIZES = ["tiny", "base", "small", "medium", "large", "large-v2", "large-v3"]
    
    def __init__(self, model_size: str = "small", output_base_dir: Optional[str] = None):
        """
        Initialize the audio transcriber.
        
        Args:
            model_size: Whisper model size to use (default: "small")
            output_base_dir: Base directory for transcripts (default: project structure)
        """
        if model_size not in self.MODEL_SIZES:
            logger.warning(f"Unknown model size '{model_size}'. Using 'small' instead.")
            model_size = "small"
            
        self.model_size = model_size
        self.model = None
        
        if output_base_dir is None:
            # Default to project structure
            self.output_base_dir = self._find_project_root() / "data" / "02_intermediate" / "transcripts"
        else:
            self.output_base_dir = Path(output_base_dir)
            
        self._ensure_dependencies()
        
    def _find_project_root(self) -> Path:
        """
        Find the project root directory.
        
        Returns:
            Path to project root directory
        """
        current = Path.cwd()
        project_name = "somali-radios-with-ai-for-food-security"
        
        while current != current.parent:
            if current.name == project_name:
                return current
            if (current / project_name).exists():
                return current / project_name
            current = current.parent
        
        # If not found, create it in the current working directory
        project_root = Path.cwd() / project_name
        logger.info(f"Project root not found, using: {project_root}")
        return project_root
    
    def _ensure_dependencies(self):
        """Ensure all required dependencies are installed."""
        self._check_whisper()
        self._check_ffmpeg()
        
    def _check_whisper(self):
        """Check if Whisper is installed and install if needed."""
        try:
            import whisper
            logger.info("OpenAI Whisper is already available")
        except ImportError:
            logger.info("Installing OpenAI Whisper...")
            subprocess.run(["pip", "install", "openai-whisper"], check=True)
            logger.info("OpenAI Whisper installed successfully")
            
    def _check_ffmpeg(self):
        """Check if ffmpeg is installed and install if needed."""
        try:
            result = subprocess.run(['ffmpeg', '-version'], 
                                  stdout=subprocess.PIPE, 
                                  stderr=subprocess.PIPE)
            if result.returncode == 0:
                logger.info("ffmpeg is available")
            else:
                raise FileNotFoundError
        except FileNotFoundError:
            logger.info("Installing ffmpeg...")
            try:
                # Try different installation methods based on the system
                subprocess.run(["apt-get", "update", "-qq"], check=True)
                subprocess.run(["apt-get", "install", "-qq", "ffmpeg"], check=True)
                logger.info("ffmpeg installed successfully via apt-get")
            except subprocess.CalledProcessError:
                logger.error("Failed to install ffmpeg. Please install it manually.")
                raise
    
    def _load_model(self):
        """Load the Whisper model if not already loaded."""
        if self.model is None:
            logger.info(f"Loading Whisper model: {self.model_size}")
            import whisper
            self.model = whisper.load_model(self.model_size)
            logger.info("Model loaded successfully")
    
    def _get_audio_files(self, input_dir: Path, audio_formats: Optional[List[str]] = None) -> List[Path]:
        """
        Get all audio files in the specified directory.
        
        Args:
            input_dir: Directory to search for audio files
            audio_formats: List of formats to search for (default: all supported)
            
        Returns:
            List of audio file paths
        """
        if audio_formats is None:
            audio_formats = self.SUPPORTED_FORMATS
            
        audio_files = []
        for fmt in audio_formats:
            pattern = input_dir / f"*.{fmt}"
            audio_files.extend(glob.glob(str(pattern)))
            
        return [Path(f) for f in sorted(audio_files)]
    
    def _convert_audio_to_wav(self, audio_file: Path, temp_dir: Path) -> Optional[Path]:
        """
        Convert audio file to WAV format for better compatibility.
        
        Args:
            audio_file: Path to the audio file to convert
            temp_dir: Temporary directory for conversion
            
        Returns:
            Path to converted WAV file if successful, None otherwise
        """
        temp_dir.mkdir(parents=True, exist_ok=True)
        temp_wav = temp_dir / f"{audio_file.stem}.wav"
        
        try:
            cmd = [
                'ffmpeg', '-y', '-i', str(audio_file),
                '-ar', '16000',  # 16kHz sample rate
                '-ac', '1',      # Mono audio
                '-c:a', 'pcm_s16le',  # 16-bit PCM
                str(temp_wav)
            ]
            
            result = subprocess.run(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            
            if result.returncode == 0:
                logger.info(f"Successfully converted to WAV: {temp_wav}")
                return temp_wav
            else:
                logger.error(f"Conversion failed: {result.stderr}")
                return None
                
        except Exception as e:
            logger.error(f"Error converting {audio_file}: {e}")
            return None
    
    def transcribe_file(self, audio_file: Path, output_dir: Path, language: str = "so") -> Optional[Path]:
        """
        Transcribe a single audio file.
        
        Args:
            audio_file: Path to the audio file
            output_dir: Directory to save the transcript
            language: Language code for transcription
            
        Returns:
            Path to transcript file if successful, None otherwise
        """
        self._load_model()
        
        base_name = audio_file.stem
        transcript_file = output_dir / f"{base_name}.txt"
        
        logger.info(f"Transcribing: {audio_file.name}")
        
        try:
            # First attempt: direct transcription
            result = self.model.transcribe(
                str(audio_file), 
                language=language, 
                fp16=False, 
                temperature=0.2,
                word_timestamps=True
            )
            
            # Save transcript
            with open(transcript_file, "w", encoding="utf-8") as f:
                f.write(result["text"].strip())
            
            logger.info(f"Transcript saved: {transcript_file}")
            return transcript_file
            
        except Exception as e:
            logger.warning(f"Direct transcription failed for {audio_file.name}: {e}")
            
            # Second attempt: convert to WAV and retry
            temp_dir = output_dir / "_temp_conversion"
            temp_wav = self._convert_audio_to_wav(audio_file, temp_dir)
            
            if temp_wav and temp_wav.exists():
                try:
                    result = self.model.transcribe(
                        str(temp_wav), 
                        language=language, 
                        fp16=False, 
                        temperature=0.2,
                        word_timestamps=True
                    )
                    
                    with open(transcript_file, "w", encoding="utf-8") as f:
                        f.write(result["text"].strip())
                    
                    logger.info(f"Transcript saved after conversion: {transcript_file}")
                    return transcript_file
                    
                except Exception as retry_e:
                    logger.error(f"Transcription failed even after conversion: {retry_e}")
            
            return None
    
    def transcribe_directory(self, input_dir: Union[str, Path], output_dir_name: Optional[str] = None,
                           language: str = "so", audio_formats: Optional[List[str]] = None) -> List[Path]:
        """
        Transcribe all audio files in a directory.
        
        Args:
            input_dir: Directory containing audio files
            output_dir_name: Custom output directory name
            language: Language code for transcription
            audio_formats: List of audio formats to process
            
        Returns:
            List of transcript file paths
        """
        input_path = Path(input_dir)
        
        if not input_path.exists():
            logger.error(f"Input directory does not exist: {input_path}")
            return []
        
        # Create output directory
        if output_dir_name is None:
            output_dir_name = f"transcripts_{input_path.name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        output_dir = self.output_base_dir / output_dir_name
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # Get audio files
        audio_files = self._get_audio_files(input_path, audio_formats)
        
        if not audio_files:
            logger.warning(f"No audio files found in {input_path}")
            return []
        
        logger.info(f"Found {len(audio_files)} audio files to transcribe")
        logger.info(f"Output directory: {output_dir}")
        
        # Transcribe files
        transcript_files = []
        failed_files = []
        
        for i, audio_file in enumerate(audio_files, 1):
            logger.info(f"Processing {i}/{len(audio_files)}: {audio_file.name}")
            
            result = self.transcribe_file(audio_file, output_dir, language)
            if result:
                transcript_files.append(result)
            else:
                failed_files.append(audio_file)
        
        # Clean up temporary conversion directory
        temp_dir = output_dir / "_temp_conversion"
        if temp_dir.exists():
            try:
                shutil.rmtree(temp_dir)
                logger.info("Removed temporary conversion directory")
            except Exception as e:
                logger.warning(f"Failed to remove temporary directory: {e}")
        
        # Log summary
        logger.info(f"\nTranscription Summary:")
        logger.info(f"Total files processed: {len(audio_files)}")
        logger.info(f"Successfully transcribed: {len(transcript_files)}")
        logger.info(f"Failed transcriptions: {len(failed_files)}")
        
        if failed_files:
            logger.warning("Failed files:")
            for file_path in failed_files:
                logger.warning(f"  - {file_path}")
        
        if transcript_files:
            logger.info("Successfully created transcripts:")
            for file_path in transcript_files:
                logger.info(f"  - {file_path}")
        
        return transcript_files


# Convenience functions for backward compatibility and ease of use
def transcribe_audio_files(input_dir: Union[str, Path], output_dir: Optional[str] = None, 
                          language: str = "so", audio_formats: Optional[List[str]] = None,
                          model_size: str = "small") -> List[Path]:
    """
    Convenience function to transcribe audio files using Whisper.
    
    Args:
        input_dir: Directory containing audio files
        output_dir: Custom output directory name
        language: Language code for transcription (default: "so" for Somali)
        audio_formats: List of audio formats to process
        model_size: Whisper model size to use
        
    Returns:
        List of transcript file paths
    """
    transcriber = AudioTranscriber(model_size=model_size)
    return transcriber.transcribe_directory(input_dir, output_dir, language, audio_formats)


def transcribe_downloaded_audio(download_dir: Union[str, Path], language: str = "so", 
                               model_size: str = "small") -> List[Path]:
    """
    Convenience function specifically for transcribing downloaded SoundCloud audio.
    Automatically detects the downloaded audio directory and creates appropriate output structure.
    
    Args:
        download_dir: Directory containing downloaded audio (relative to project data/01_raw/comparison/)
        language: Language code for transcription
        model_size: Whisper model size to use
        
    Returns:
        List of transcript file paths
    """
    # Find project root and construct full path
    transcriber = AudioTranscriber(model_size=model_size)
    project_root = transcriber._find_project_root()
    
    # If download_dir is just a directory name, assume it's in the comparison folder
    download_path = Path(download_dir)
    if not download_path.is_absolute():
        download_path = project_root / "data" / "01_raw" / "comparison" / download_dir
    
    if not download_path.exists():
        logger.error(f"Download directory not found: {download_path}")
        return []
    
    # Create output directory name based on input
    output_name = f"transcripts_{download_path.name}"
    
    return transcriber.transcribe_directory(download_path, output_name, language)


#### 📋 INSTRUCTION
**Run the cell below to use Whisper to transcribe all Somali audio files in the specified directory and optionally save the transcripts to Google Drive.**

In [4]:
# Example usage and main execution
if __name__ == "__main__":
    # Example 1: Transcribe the downloaded SoundCloud files
    input_directory = "soundcloud_2025-03-15_to_2025-03-16"
    
    transcripts = transcribe_downloaded_audio(
        download_dir=input_directory,
        language="so",  # Somali language code
        model_size="large"  # Options: tiny, base, small, medium, large
    )
    
    print(f"\nTranscription completed!")
    print(f"Created {len(transcripts)} transcript files:")
    for transcript in transcripts:
        print(f"  - {transcript}")
    
    # Example 2: Direct usage with full path
    # transcripts = transcribe_audio_files(
    #     input_dir="/full/path/to/audio/directory",
    #     language="so",
    #     model_size="medium"
    # )
    
    # Example 3: Using the class directly for more control
    # transcriber = AudioTranscriber(model_size="large")
    # transcripts = transcriber.transcribe_directory(
    #     input_dir="path/to/audio",
    #     output_dir_name="custom_transcripts",
    #     language="so"
    # )

2025-09-22 12:38:39,496 - INFO - OpenAI Whisper is already available
2025-09-22 12:38:39,546 - INFO - ffmpeg is available
2025-09-22 12:38:39,550 - INFO - Found 2 audio files to transcribe
2025-09-22 12:38:39,551 - INFO - Output directory: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/transcripts_soundcloud_2025-03-15_to_2025-03-16
2025-09-22 12:38:39,552 - INFO - Processing 1/2: IDAACADDA 15-MAR-2025.mp3
2025-09-22 12:38:39,553 - INFO - Loading Whisper model: large
100%|██████████████████████████████████████| 2.88G/2.88G [00:30<00:00, 100MiB/s]
2025-09-22 12:39:32,820 - INFO - Model loaded successfully
2025-09-22 12:39:32,905 - INFO - Transcribing: IDAACADDA 15-MAR-2025.mp3


KeyboardInterrupt: 

### Performance Overview  

In [5]:
"""
Somali Transcript Analysis Tool

A comprehensive tool for analyzing Somali language transcripts with linguistic metrics,
pattern detection, and quality assessment capabilities.
"""

import os
import re
import textwrap
import logging
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Union
from dataclasses import dataclass

# Optional NLTK import with graceful fallback
try:
    import nltk
    from nltk.tokenize import sent_tokenize, word_tokenize
    NLTK_AVAILABLE = True
    
    # Download required NLTK data if needed
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt', quiet=True)
        
except ImportError:
    NLTK_AVAILABLE = False
    logging.info("NLTK not available. Some advanced features will be disabled.")


# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


@dataclass
class TranscriptStats:
    """Data class to hold transcript analysis statistics."""
    num_lines: int
    total_words: int
    total_characters: int
    avg_words_per_line: float
    avg_word_length: float
    avg_chars_per_line: float
    repetition_count: int
    estimated_sentences: int
    line_length_range: Tuple[int, int]
    most_common_words: List[Tuple[str, int]]
    repeated_segments: List[Tuple[str, int]]


class SomaliTranscriptAnalyzer:
    """
    A comprehensive analyzer for Somali language transcripts.
    
    Provides linguistic analysis, quality metrics, and pattern detection
    specifically designed for Somali radio transcripts.
    """
    
    # Common Somali stop words and particles
    SOMALI_STOP_WORDS = {
        'ah', 'aha', 'waa', 'waxaa', 'baa', 'ayaa', 'oo', 'iyo', 'ama', 'laakiin',
        'si', 'ka', 'ku', 'la', 'uu', 'ay', 'aan', 'aad', 'uu', 'ee', 'oo',
        'in', 'an', 'soo', 'wax', 'ma', 'mise', 'haddii', 'markii', 'intii'
    }
    
    # Somali sentence-ending patterns
    SENTENCE_ENDINGS = ['.', '!', '?', ':', ';']
    
    def __init__(self, project_root: Optional[Path] = None):
        """
        Initialize the Somali transcript analyzer.
        
        Args:
            project_root: Path to project root (auto-detected if None)
        """
        if project_root is None:
            self.project_root = self._find_project_root()
        else:
            self.project_root = Path(project_root)
            
        self.transcripts_dir = self.project_root / "data" / "02_intermediate" / "transcripts"
        self.analysis_output_dir = self.project_root / "data" / "03_primary" / "analysis"
        self.analysis_output_dir.mkdir(parents=True, exist_ok=True)
        
    def _find_project_root(self) -> Path:
        """Find the project root directory."""
        current = Path.cwd()
        project_name = "somali-radios-with-ai-for-food-security"
        
        while current != current.parent:
            if current.name == project_name:
                return current
            if (current / project_name).exists():
                return current / project_name
            current = current.parent
        
        # If not found, use current working directory
        logger.warning(f"Project root not found, using current directory: {Path.cwd()}")
        return Path.cwd()
    
    def _read_transcript_file(self, file_path: Path, max_lines: Optional[int] = None) -> List[str]:
        """
        Read transcript file and return non-empty lines.
        
        Args:
            file_path: Path to transcript file
            max_lines: Maximum number of lines to read
            
        Returns:
            List of non-empty lines
        """
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                if max_lines:
                    lines = [file.readline().strip() for _ in range(max_lines)]
                else:
                    lines = [line.strip() for line in file.readlines()]
                    
            # Remove empty lines
            lines = [line for line in lines if line.strip()]
            return lines
            
        except FileNotFoundError:
            logger.error(f"File not found: {file_path}")
            raise
        except Exception as e:
            logger.error(f"Error reading file {file_path}: {e}")
            raise
    
    def _calculate_basic_stats(self, lines: List[str]) -> Dict[str, Any]:
        """Calculate basic statistical metrics for the transcript."""
        if not lines:
            return {}
            
        num_lines = len(lines)
        total_chars = sum(len(line) for line in lines)
        total_words = sum(len(line.split()) for line in lines)
        
        return {
            'num_lines': num_lines,
            'total_words': total_words,
            'total_characters': total_chars,
            'avg_words_per_line': total_words / num_lines,
            'avg_word_length': sum(len(word) for line in lines for word in line.split()) / total_words if total_words > 0 else 0,
            'avg_chars_per_line': total_chars / num_lines
        }
    
    def _analyze_word_frequency(self, lines: List[str], top_n: int = 20) -> List[Tuple[str, int]]:
        """Analyze word frequency in the transcript."""
        # Combine all lines and normalize
        all_text = ' '.join(lines).lower()
        # Remove punctuation and split into words
        words = re.findall(r'\b\w+\b', all_text)
        
        # Filter out very short words and common stop words
        filtered_words = [
            word for word in words 
            if len(word) > 2 and word not in self.SOMALI_STOP_WORDS
        ]
        
        word_freq = Counter(filtered_words)
        return word_freq.most_common(top_n)
    
    def _detect_repetitions(self, lines: List[str]) -> int:
        """Count adjacent word repetitions in the transcript."""
        repetition_count = 0
        for line in lines:
            words = line.split()
            for i in range(len(words) - 1):
                if words[i].lower() == words[i+1].lower():
                    repetition_count += 1
        return repetition_count
    
    def _find_repeated_segments(self, lines: List[str], min_occurrences: int = 3) -> List[Tuple[str, int]]:
        """Find commonly repeated word segments in the transcript."""
        text = ' '.join(lines).lower()
        words = text.split()
        repeated_segments = []
        
        # Check for segments of different lengths
        for segment_len in range(2, 6):  # 2 to 5 words
            if len(words) < segment_len:
                continue
                
            segments = [
                ' '.join(words[i:i+segment_len]) 
                for i in range(len(words) - segment_len + 1)
            ]
            
            segment_counts = Counter(segments)
            repeated = [
                (segment, count) for segment, count in segment_counts.items() 
                if count >= min_occurrences and len(segment.strip()) > 5
            ]
            repeated_segments.extend(repeated)
        
        # Sort by frequency and return top results
        return sorted(repeated_segments, key=lambda x: x[1], reverse=True)[:15]
    
    def _estimate_sentences(self, lines: List[str]) -> Tuple[int, List[str]]:
        """Estimate sentence count and extract sample sentences."""
        sentences = []
        current_sentence = ""
        
        for line in lines:
            # Split by sentence-ending punctuation
            parts = re.split(r'[.!?:;]', line)
            
            for i, part in enumerate(parts):
                if part.strip():
                    if current_sentence:
                        current_sentence += " " + part.strip()
                    else:
                        current_sentence = part.strip()
                    
                    # If this part was followed by punctuation, end the sentence
                    if i < len(parts) - 1 or any(line.rstrip().endswith(p) for p in self.SENTENCE_ENDINGS):
                        if current_sentence:
                            sentences.append(current_sentence)
                            current_sentence = ""
        
        # Add any remaining sentence
        if current_sentence:
            sentences.append(current_sentence)
        
        return len(sentences), sentences[:10]  # Return count and first 10 sentences
    
    def _analyze_line_lengths(self, lines: List[str]) -> Tuple[int, int, List[int]]:
        """Analyze line length distribution."""
        line_lengths = [len(line.split()) for line in lines]
        return min(line_lengths), max(line_lengths), line_lengths
    
    def _assess_transcript_quality(self, stats: TranscriptStats) -> Dict[str, Any]:
        """Assess the quality of the transcript based on various metrics."""
        quality_issues = []
        quality_score = 100  # Start with perfect score
        
        # Check for excessive repetitions
        repetition_ratio = stats.repetition_count / stats.total_words if stats.total_words > 0 else 0
        if repetition_ratio > 0.1:  # More than 10% repetitions
            quality_issues.append(f"High repetition rate: {repetition_ratio:.2%}")
            quality_score -= 20
        
        # Check average word length (very short might indicate poor transcription)
        if stats.avg_word_length < 3:
            quality_issues.append(f"Short average word length: {stats.avg_word_length:.2f}")
            quality_score -= 15
        
        # Check line length variance (too uniform might indicate issues)
        min_len, max_len = stats.line_length_range
        if max_len - min_len < 5 and stats.num_lines > 10:
            quality_issues.append("Low variance in line lengths")
            quality_score -= 10
        
        # Check for very short lines that might indicate transcription errors
        if min_len < 2:
            quality_issues.append("Very short lines detected")
            quality_score -= 10
        
        return {
            'quality_score': max(0, quality_score),  # Don't go below 0
            'quality_issues': quality_issues,
            'assessment': 'Good' if quality_score >= 80 else 'Fair' if quality_score >= 60 else 'Poor'
        }
    
    def analyze_transcript(self, file_path: Union[str, Path], max_lines: Optional[int] = None, 
                          sample_size: int = 10, save_analysis: bool = True) -> TranscriptStats:
        """
        Perform comprehensive analysis of a Somali transcript.
        
        Args:
            file_path: Path to transcript file (absolute or relative to transcripts directory)
            max_lines: Maximum number of lines to analyze
            sample_size: Number of sample lines to display
            save_analysis: Whether to save analysis results to file
            
        Returns:
            TranscriptStats object with analysis results
        """
        # Handle path resolution
        file_path = Path(file_path)
        if not file_path.is_absolute():
            # Try relative to transcripts directory first
            if (self.transcripts_dir / file_path).exists():
                file_path = self.transcripts_dir / file_path
            # Try relative to current directory
            elif not file_path.exists():
                # Try finding it in any subdirectory of transcripts
                for transcript_file in self.transcripts_dir.rglob(file_path.name):
                    file_path = transcript_file
                    break
        
        if not file_path.exists():
            raise FileNotFoundError(f"Transcript file not found: {file_path}")
        
        logger.info(f"Analyzing transcript: {file_path.name}")
        
        # Read transcript
        lines = self._read_transcript_file(file_path, max_lines)
        
        if not lines:
            logger.warning("No content found in transcript file")
            return None
        
        # Perform analysis
        basic_stats = self._calculate_basic_stats(lines)
        most_common = self._analyze_word_frequency(lines)
        repetitions = self._detect_repetitions(lines)
        repeated_segments = self._find_repeated_segments(lines)
        sentence_count, sample_sentences = self._estimate_sentences(lines)
        min_len, max_len, line_lengths = self._analyze_line_lengths(lines)
        
        # Create stats object
        stats = TranscriptStats(
            num_lines=basic_stats['num_lines'],
            total_words=basic_stats['total_words'],
            total_characters=basic_stats['total_characters'],
            avg_words_per_line=basic_stats['avg_words_per_line'],
            avg_word_length=basic_stats['avg_word_length'],
            avg_chars_per_line=basic_stats['avg_chars_per_line'],
            repetition_count=repetitions,
            estimated_sentences=sentence_count,
            line_length_range=(min_len, max_len),
            most_common_words=most_common,
            repeated_segments=repeated_segments
        )
        
        # Quality assessment
        quality_info = self._assess_transcript_quality(stats)
        
        # Display results
        self._display_analysis_results(file_path.name, lines, stats, quality_info, 
                                     sample_sentences, sample_size)
        
        # Save analysis if requested
        if save_analysis:
            self._save_analysis_report(file_path, stats, quality_info, sample_sentences)
        
        return stats
    
    def _display_analysis_results(self, filename: str, lines: List[str], stats: TranscriptStats,
                                 quality_info: Dict[str, Any], sample_sentences: List[str],
                                 sample_size: int):
        """Display comprehensive analysis results."""
        print(f"\n{'='*60}")
        print(f"SOMALI TRANSCRIPT ANALYSIS: {filename}")
        print(f"{'='*60}")
        
        # Sample lines
        print(f"\n{'='*20} SAMPLE LINES {'='*20}")
        sample_indices = list(range(min(sample_size, len(lines))))
        for i in sample_indices:
            wrapped_text = textwrap.fill(lines[i], width=80)
            print(f"Line {i+1}: {wrapped_text}")
        
        # Basic statistics
        print(f"\n{'='*20} BASIC STATISTICS {'='*20}")
        print(f"Total lines: {stats.num_lines:,}")
        print(f"Total words: {stats.total_words:,}")
        print(f"Total characters: {stats.total_characters:,}")
        print(f"Average words per line: {stats.avg_words_per_line:.2f}")
        print(f"Average word length: {stats.avg_word_length:.2f} characters")
        print(f"Average characters per line: {stats.avg_chars_per_line:.1f}")
        print(f"Estimated sentences: {stats.estimated_sentences:,}")
        print(f"Adjacent word repetitions: {stats.repetition_count}")
        
        # Line length analysis
        min_len, max_len = stats.line_length_range
        print(f"\nLine length range: {min_len} - {max_len} words")
        
        # Quality assessment
        print(f"\n{'='*20} QUALITY ASSESSMENT {'='*20}")
        print(f"Quality score: {quality_info['quality_score']}/100 ({quality_info['assessment']})")
        if quality_info['quality_issues']:
            print("Quality issues detected:")
            for issue in quality_info['quality_issues']:
                print(f"  - {issue}")
        else:
            print("No significant quality issues detected.")
        
        # Most common words
        print(f"\n{'='*20} MOST COMMON WORDS {'='*20}")
        for word, count in stats.most_common_words[:15]:
            percentage = (count / stats.total_words) * 100
            print(f"{word:15} {count:6,} ({percentage:.1f}%)")
        
        # Repeated segments
        if stats.repeated_segments:
            print(f"\n{'='*20} REPEATED SEGMENTS {'='*20}")
            for segment, count in stats.repeated_segments[:10]:
                print(f"'{segment}' → {count} times")
        
        # Sample sentences
        if sample_sentences:
            print(f"\n{'='*20} SAMPLE SENTENCES {'='*20}")
            for i, sentence in enumerate(sample_sentences[:5], 1):
                wrapped = textwrap.fill(sentence, width=75)
                print(f"{i}. {wrapped}")
    
    def _save_analysis_report(self, file_path: Path, stats: TranscriptStats, 
                             quality_info: Dict[str, Any], sample_sentences: List[str]):
        """Save analysis report to file."""
        report_name = f"analysis_{file_path.stem}_{file_path.parent.name}.txt"
        report_path = self.analysis_output_dir / report_name
        
        try:
            with open(report_path, 'w', encoding='utf-8') as f:
                f.write(f"Somali Transcript Analysis Report\n")
                f.write(f"Generated: {file_path}\n")
                f.write(f"{'='*50}\n\n")
                
                # Write all the analysis data
                f.write(f"BASIC STATISTICS\n")
                f.write(f"Lines: {stats.num_lines:,}\n")
                f.write(f"Words: {stats.total_words:,}\n")
                f.write(f"Characters: {stats.total_characters:,}\n")
                f.write(f"Avg words/line: {stats.avg_words_per_line:.2f}\n")
                f.write(f"Avg word length: {stats.avg_word_length:.2f}\n")
                f.write(f"Estimated sentences: {stats.estimated_sentences:,}\n\n")
                
                f.write(f"QUALITY ASSESSMENT\n")
                f.write(f"Score: {quality_info['quality_score']}/100 ({quality_info['assessment']})\n")
                if quality_info['quality_issues']:
                    f.write("Issues:\n")
                    for issue in quality_info['quality_issues']:
                        f.write(f"  - {issue}\n")
                f.write("\n")
                
                # Save common words and repeated segments
                f.write("MOST COMMON WORDS\n")
                for word, count in stats.most_common_words:
                    f.write(f"{word}: {count}\n")
                
                if stats.repeated_segments:
                    f.write("\nREPEATED SEGMENTS\n")
                    for segment, count in stats.repeated_segments:
                        f.write(f"'{segment}': {count}\n")
            
            logger.info(f"Analysis report saved: {report_path}")
            
        except Exception as e:
            logger.error(f"Failed to save analysis report: {e}")
    
    def analyze_directory(self, transcript_dir: Optional[Union[str, Path]] = None,
                         file_pattern: str = "*.txt") -> Dict[str, TranscriptStats]:
        """
        Analyze all transcript files in a directory.
        
        Args:
            transcript_dir: Directory containing transcript files (default: auto-detect)
            file_pattern: File pattern to match (default: *.txt)
            
        Returns:
            Dictionary mapping filenames to analysis results
        """
        if transcript_dir is None:
            # Find the most recent transcript directory
            transcript_dirs = [d for d in self.transcripts_dir.iterdir() if d.is_dir()]
            if not transcript_dirs:
                logger.error("No transcript directories found")
                return {}
            transcript_dir = max(transcript_dirs, key=lambda d: d.stat().st_mtime)
        else:
            transcript_dir = Path(transcript_dir)
            if not transcript_dir.is_absolute():
                transcript_dir = self.transcripts_dir / transcript_dir
        
        logger.info(f"Analyzing all transcripts in: {transcript_dir}")
        
        # Find all transcript files
        transcript_files = list(transcript_dir.glob(file_pattern))
        
        if not transcript_files:
            logger.warning(f"No transcript files found matching '{file_pattern}' in {transcript_dir}")
            return {}
        
        results = {}
        for transcript_file in transcript_files:
            try:
                stats = self.analyze_transcript(transcript_file, save_analysis=True)
                results[transcript_file.name] = stats
            except Exception as e:
                logger.error(f"Failed to analyze {transcript_file.name}: {e}")
        
        logger.info(f"Completed analysis of {len(results)} transcript files")
        return results


# Convenience functions for easy usage
def analyze_somali_transcript(file_path: Union[str, Path], max_lines: Optional[int] = None,
                             sample_size: int = 10) -> TranscriptStats:
    """
    Convenience function to analyze a single Somali transcript.
    
    Args:
        file_path: Path to transcript file
        max_lines: Maximum number of lines to analyze
        sample_size: Number of sample lines to display
        
    Returns:
        TranscriptStats object with analysis results
    """
    analyzer = SomaliTranscriptAnalyzer()
    return analyzer.analyze_transcript(file_path, max_lines, sample_size)


def analyze_latest_transcripts(sample_size: int = 10) -> Dict[str, TranscriptStats]:
    """
    Convenience function to analyze the most recently created transcript directory.
    
    Args:
        sample_size: Number of sample lines to display per file
        
    Returns:
        Dictionary mapping filenames to analysis results
    """
    analyzer = SomaliTranscriptAnalyzer()
    return analyzer.analyze_directory()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


#### 📋 INSTRUCTION
Run the cell below to visualize the transcript generated by Whisper.

In [6]:
# Example usage
if __name__ == "__main__":
    # Example 1: Analyze specific transcript files
    transcript_files = [
        "transcripts_soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 15-MAR-2025.txt",
        "transcripts_soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 16-MAR-2025.txt"
    ]
    
    for file_path in transcript_files:
        try:
            results = analyze_somali_transcript(file_path)
            print(f"\nAnalysis completed for: {file_path}")
            print(f"Quality score: {results} (if stats returned)")
        except Exception as e:
            print(f"Error analyzing {file_path}: {e}")
    
    # Example 2: Analyze all transcripts in the latest directory
    # all_results = analyze_latest_transcripts()
    # print(f"\nAnalyzed {len(all_results)} transcript files")
    
    # Example 3: Use the class directly for more control
    # analyzer = SomaliTranscriptAnalyzer()
    # results = analyzer.analyze_directory("transcripts_soundcloud_2025-03-15_to_2025-03-16")

Analyzing file: IDAACADDA 15-MAR-2025.txt

==== Sample Lines ====
Line 1: موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع
موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع موضوع مو

### Problem ❌
OpenAI's Whisper model detects Somali (`so`) but **transcribes it in Arabic script** instead of **Somali Latin script**. This makes the output unusable for Somali language.  

## Observations  
- Produces Arabic text instead of Latin script.  
- Users report **repeated phrases** and **incorrect words**.  
- Confirmed in multiple discussions:  
  - [GitHub Issue #234](https://github.com/openai/whisper/discussions/234)  
  - [GitHub Issue #2110](https://github.com/openai/whisper/discussions/2110)  

## Possible Causes  
- Whisper **lacks training on Somali Latin script**.  

## Whisper Small Somali (Specialized Model)

### Why?  
OpenAI’s Whisper model **transcribes Somali in Arabic script** instead of **Latin script**, making it unsuitable for Somali. Additionally, it produces **inaccurate and repetitive transcriptions**.

### Solution
We are switching to **Hugging Face’s Whisper model trained specifically for Somali**:  
➡️ **Model:** `steja/whisper-small-somali`  
➡️ **Benefit:** Ensures accurate Somali transcription in **Latin script**  

### Reference  
📄 [Hugging Face Model Docs](https://huggingface.co/steja/whisper-small-somali)


In [7]:
def transcribe_audio_files(input_dir, output_dir=None, language="so", save_to_drive=False,
                        audio_formats=None):
    """
    Transcribe audio files using HuggingFace's Whisper model trained for Somali.

    Args:
        input_dir: Directory containing the audio files
        output_dir: Directory to save transcripts (default: input_dir + "_transcripts")
        language: Language code for transcription (default: "so" for Somali)
        save_to_drive: Whether to save transcripts to Google Drive (default: False)
        audio_formats: List of audio formats to process (default: ["wav", "mp3", "m4a", "flac", "ogg"])

    Returns:
        List of transcript file paths
    """
    # Install required packages if not already installed
    try:
        import transformers
    except ImportError:
        print("Installing transformers...")
        !pip install -q transformers
        import transformers

    try:
        import torch
    except ImportError:
        print("Installing PyTorch...")
        !pip install -q torch
        import torch

    try:
        import torchaudio
    except ImportError:
        print("Installing torchaudio...")
        !pip install -q torchaudio
        import torchaudio

    try:
        import soundfile
    except ImportError:
        print("Installing soundfile for additional audio format support...")
        !pip install -q soundfile
        import soundfile

    import os
    import glob
    from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor

    # Define default audio formats if none provided
    if audio_formats is None:
        audio_formats = ["wav", "mp3", "m4a", "flac", "ogg"]

    # Ensure formats have the dot prefix for glob patterns
    formats_pattern = [f"*.{fmt}" for fmt in audio_formats]

    # Set output directory
    if not output_dir:
        output_dir = "whisper_small_somali_transcripts_" + input_dir

    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Loading Hugging Face Whisper model (steja/whisper-small-somali)...")
    # Load the Whisper model from Hugging Face
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load model and processor
    model_id = "steja/whisper-small-somali"
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id)
    processor = AutoProcessor.from_pretrained(model_id)

    # Move model to appropriate device
    model.to(device)

    # Create the pipeline for automatic speech recognition
    transcriber = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        max_new_tokens=128,
        chunk_length_s=30,
        batch_size=16,
        device=device
    )

    # Get all audio files with specified formats in the input directory
    audio_files = []
    for pattern in formats_pattern:
        audio_files.extend(glob.glob(os.path.join(input_dir, pattern)))

    if not audio_files:
        print(f"No audio files with formats {audio_formats} found in {input_dir}")
        return []

    print(f"Found {len(audio_files)} audio files to transcribe.")
    print(f"Formats to process: {', '.join(audio_formats)}")

    transcript_files = []

    for audio_file in audio_files:
        filename = os.path.basename(audio_file)
        base_name = os.path.splitext(filename)[0]
        transcript_file = os.path.join(output_dir, f"{base_name}.txt")
        file_format = os.path.splitext(filename)[1][1:].lower()  # Get format without dot

        print(f"\nTranscribing: {filename} (Format: {file_format})")

        try:
            # Transcribe audio without requesting timestamps
            result = transcriber(audio_file)

            # Extract the transcript text
            if isinstance(result, dict) and "text" in result:
                transcript_text = result["text"]
            else:
                # Handle different return formats
                transcript_text = result

            # Save full transcript
            with open(transcript_file, "w", encoding="utf-8") as f:
                f.write(transcript_text)

            print(f"Transcript saved to: {transcript_file}")
            transcript_files.append(transcript_file)

        except Exception as e:
            print(f"Error transcribing {filename}: {str(e)}")

            # If there's an error with non-WAV formats, try to convert to WAV first
            if file_format != "wav":
                try:
                    print(f"Attempting to convert {file_format} to WAV format and retry...")

                    # Create a temporary directory for conversion if it doesn't exist
                    temp_dir = os.path.join(input_dir, "_temp_conversion")
                    if not os.path.exists(temp_dir):
                        os.makedirs(temp_dir)

                    temp_wav = os.path.join(temp_dir, f"{base_name}.wav")

                    # Load and resave as WAV using torchaudio
                    try:
                        waveform, sample_rate = torchaudio.load(audio_file)
                        torchaudio.save(temp_wav, waveform, sample_rate)
                        print(f"Successfully converted to WAV: {temp_wav}")

                        # Try transcription again with the WAV file
                        result = transcriber(temp_wav)

                        if isinstance(result, dict) and "text" in result:
                            transcript_text = result["text"]
                        else:
                            transcript_text = result

                        with open(transcript_file, "w", encoding="utf-8") as f:
                            f.write(transcript_text)

                        print(f"Transcript saved to: {transcript_file} (after conversion)")
                        transcript_files.append(transcript_file)

                    except Exception as conv_err:
                        print(f"Conversion failed: {str(conv_err)}")

                except Exception as retry_err:
                    print(f"Retry failed: {str(retry_err)}")

    # Clean up temporary conversion directory if it exists
    temp_dir = os.path.join(input_dir, "_temp_conversion")
    if os.path.exists(temp_dir):
        import shutil
        try:
            shutil.rmtree(temp_dir)
            print(f"Removed temporary conversion directory: {temp_dir}")
        except Exception as e:
            print(f"Failed to remove temporary directory: {str(e)}")

    # Save to Google Drive if requested
    if save_to_drive and transcript_files:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            print("Google Drive mounted successfully")
        except:
            print("Not running in Google Colab or Drive already mounted")
        drive_path = "/content/drive/MyDrive/" + os.path.basename(output_dir)
        print(f"\nSaving transcripts to Google Drive at: {drive_path}")

        # Create directory in Drive if it doesn't exist
        if not os.path.exists(drive_path):
            os.makedirs(drive_path)

        # Copy files to Drive
        os.system(f"cp -r {output_dir}/* {drive_path}/")
        print(f"Transcripts successfully saved to Google Drive")

    print(f"\nTranscription Summary:")
    print(f"Total files processed: {len(audio_files)}")
    print(f"Transcripts created: {len(transcript_files)}")

    return transcript_files

#### 📋 INSTRUCTION   
**Run the cell below to use Whisper Small Somali to transcribe all Somali audio files in the specified directory and optionally save the transcripts to Google Drive.**

In [8]:
# Example usage
input_directory = "soundcloud_2025-03-15_to_2025-03-16"

# Transcribe all audio files in the directory
transcripts = transcribe_audio_files(
    input_dir=input_directory,
    language="so",  # Somali language code
    save_to_drive=True  # Change to True to save to Google Drive
)

## Evaluation of Whisper Small Somali (Specialized Model)

### Transcription

#### 📋 INSTRUCTION
Run the cell below to visualize the transcript generated this time by Whisper Small Somali.

In [9]:
# Example usage
file_path = "/content/whisper_small_somali_transcripts_soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 15-MAR-2025.txt"
# file_path = "/content/whisper_small_somali_transcripts_soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 16-MAR-2025.txt"

results = analyze_somali_transcript(file_path)

Analyzing file: IDAACADDA 15-MAR-2025.txt
Error: File not found at /content/whisper_small_somali_transcripts_soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 15-MAR-2025.txt


## Gemini Flash 2.0 – Somali Transcription Update

### Why?  
After achieving 45.9% WER with the top-performing Scribe v1 model on the FLEURS benchmark, Gemini Flash 2.0 ranks as the second-best with a 54.4% Word Error Rate (WER).
🔗 [See Benchmark Results](https://elevenlabs.io/speech-to-text/somali)

### Solution
We're now deploying Gemini's in-house Somali transcription model, designed specifically for high accuracy and Latin script output:
➡️ **Model:**  Gemini Flash 2.0 Somali
➡️ **Benefit:** Improved Somali transcription quality using Gemini-tuned architecture, optimized for real-world use cases.

### Reference  
📄 [Gemini Flash 2.0 documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/2-0-flash)

In [10]:
import os
import glob
from typing import Optional, List, Dict
from google import genai
from google.genai import types
from datetime import datetime
import json
import time
import subprocess # For ffmpeg
import shutil     # For directory cleanup

def transcribe_audio_files(
    input_dir: str,
    api_key: str, # API key is now passed as an argument
    output_dir: Optional[str] = None,
    language: str = "so", # Language hint for Gemini, not a strict filter
    save_to_drive: bool = False,
    audio_formats: Optional[List[str]] = None,
    model: str = "gemini-2.0-flash",
    retry_count: int = 3,
    delay_between_failures: int = 10
) -> List[str]:
    """
    Transcribe audio files using Google Gemini Flash 2.0.

    Args:
        input_dir (str): Directory containing audio files.
        api_key (str): Your Google Gemini API key.
        output_dir (str, optional): Directory to save transcripts.
                                    Defaults to input_dir + "_gemini_transcripts".
        language (str, optional): Language code for transcription (e.g., "so" for Somali).
                                  This acts as a hint for the model. Defaults to "so".
        save_to_drive (bool, optional): Whether to save transcripts to Google Drive.
                                        Defaults to False.
        audio_formats (list, optional): List of audio formats to process.
                                        Defaults to ["wav", "mp3", "m4a", "flac", "ogg"].
        model (str, optional): The Gemini model to use for transcription.
                               Defaults to "gemini-2.0-flash".
        retry_count (int, optional): Number of times to retry failed transcriptions.
                                     Defaults to 3.
        delay_between_failures (int, optional): Seconds to wait between retry attempts.
                                                Defaults to 10.

    Returns:
        List[str]: List of transcript file paths created.
    """
    if not api_key:
        raise EnvironmentError("API key cannot be empty. Please provide your Gemini API key.")

    # Mount Google Drive at the beginning if save_to_drive is True
    if save_to_drive:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            print("Google Drive mounted successfully")
        except ImportError:
            print("Not running in Google Colab or Drive module not available")
            save_to_drive = False # Disable if not in Colab
        except Exception as e:
            print(f"Error mounting Google Drive: {str(e)}")
            print("Continuing without Google Drive. Files will only be saved locally.")
            save_to_drive = False  # Disable save_to_drive if mounting fails

    # Check and install ffmpeg if needed (required for handling different audio formats)
    try:
        subprocess.run(['ffmpeg', '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        print("ffmpeg is already installed.")
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("Installing ffmpeg (required for audio format conversion)...")
        # For Colab/Debian-based systems
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-qq', 'ffmpeg'], check=True)
        print("ffmpeg installed successfully.")

    # Initialize Gemini client
    client = genai.Client(api_key=api_key)

    # Define default audio formats if none provided
    if audio_formats is None:
        audio_formats = ["wav", "mp3", "m4a", "flac", "ogg"]

    # Ensure formats have the dot prefix for glob patterns
    formats_pattern = [f"*.{fmt}" for fmt in audio_formats]

    # Set output directory
    if not output_dir:
        output_dir = "gemini_transcripts_" + os.path.basename(input_dir.rstrip('/'))

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    print(f"Using Gemini model: {model}")

    # Get all audio files with specified formats in the input directory
    audio_files = []
    for pattern in formats_pattern:
        audio_files.extend(glob.glob(os.path.join(input_dir, pattern)))

    if not audio_files:
        print(f"No audio files with formats {audio_formats} found in {input_dir}")
        return []

    print(f"Found {len(audio_files)} audio files to transcribe.")
    print(f"Formats to process: {', '.join(audio_formats)}")

    transcript_files = []
    temp_conversion_dir = os.path.join(input_dir, "_temp_gemini_conversion")
    os.makedirs(temp_conversion_dir, exist_ok=True) # Create temp dir once

    for audio_file in audio_files:
        filename = os.path.basename(audio_file)
        base_name = os.path.splitext(filename)[0]
        file_extension = os.path.splitext(filename)[1][1:].lower() # Get format without dot
        transcript_file = os.path.join(output_dir, f"{base_name}.txt")

        # Skip if already transcribed
        if os.path.exists(transcript_file):
            print(f"Skipping {filename} - already transcribed")
            transcript_files.append(transcript_file) # Add to list even if skipped
            continue

        print(f"\nTranscribing: {filename} (Format: {file_extension})")

        current_audio_path = audio_file
        needs_conversion = False

        if file_extension != "mp3":
            needs_conversion = True
            temp_mp3_path = os.path.join(temp_conversion_dir, f"{base_name}.mp3")
            print(f"Converting {file_extension} to MP3: {audio_file} -> {temp_mp3_path}")
            try:
                cmd = [
                    'ffmpeg', '-y', '-i', audio_file,
                    '-vn', # no video
                    '-acodec', 'libmp3lame', # encode to mp3
                    '-ar', '16000', # 16 kHz sample rate (common for speech)
                    '-ac', '1', # mono audio
                    '-b:a', '32k', # bitrate, adjust as needed for quality vs size
                    temp_mp3_path
                ]
                process = subprocess.run(
                    cmd,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE,
                    check=True # Raise CalledProcessError if command fails
                )
                current_audio_path = temp_mp3_path
                print(f"Successfully converted to MP3.")
            except subprocess.CalledProcessError as e:
                print(f"FFmpeg conversion failed for {filename}: {e.stderr.decode()}")
                print(f"Skipping {filename} due to conversion error.")
                # Log failure
                with open(os.path.join(output_dir, "failed_transcriptions.json"), "a") as f:
                    failure_info = {
                        "filename": filename,
                        "path": audio_file,
                        "timestamp": datetime.now().isoformat(),
                        "error": f"FFmpeg conversion failed: {e.stderr.decode()}"
                    }
                    f.write(json.dumps(failure_info) + "\n")
                continue # Skip to next audio file

        # Try transcription with retries
        for attempt in range(retry_count):
            try:
                # Read audio file bytes (from original or converted MP3)
                with open(current_audio_path, "rb") as f:
                    audio_bytes = f.read()

                # Create a Part with correct MIME type
                audio_part = types.Part.from_bytes(
                    data=audio_bytes,
                    mime_type="audio/mp3"
                )

                # Create prompt - include language hint
                prompt = f"Generate a transcript of the speech in {language} language."

                # Request transcript
                response = client.models.generate_content(
                    model=model,
                    contents=[prompt, audio_part]
                )

                # Return the transcript
                if hasattr(response, 'text'):
                    transcript_text = response.text
                else:
                    raise ValueError("No transcript text found in the response.")

                # Save the transcript
                with open(transcript_file, "w", encoding="utf-8") as f:
                    f.write(transcript_text)

                print(f"Successfully transcribed {filename}")
                transcript_files.append(transcript_file)
                break # Exit retry loop on success

            except Exception as e:
                print(f"Attempt {attempt+1}/{retry_count} failed for {filename}: {str(e)}")
                if attempt < retry_count - 1:
                    print(f"Waiting {delay_between_failures} seconds before retrying...")
                    time.sleep(delay_between_failures)
                else:
                    print(f"All attempts failed for {filename}")
                    # Log failure
                    with open(os.path.join(output_dir, "failed_transcriptions.json"), "a") as f:
                        failure_info = {
                            "filename": filename,
                            "path": audio_file,
                            "timestamp": datetime.now().isoformat(),
                            "error": str(e)
                        }
                        f.write(json.dumps(failure_info) + "\n")

    # Clean up temporary conversion directory
    if os.path.exists(temp_conversion_dir):
        try:
            shutil.rmtree(temp_conversion_dir)
            print(f"Removed temporary conversion directory: {temp_conversion_dir}")
        except Exception as e:
            print(f"Failed to remove temporary directory: {str(e)}")

    # Save to Google Drive if requested
    if save_to_drive and transcript_files:
        # Use the same directory name for Google Drive saving
        drive_dir_name = os.path.basename(output_dir.rstrip('/'))
        drive_path = os.path.join("/content/drive/MyDrive/", drive_dir_name)
        print(f"\nSaving transcripts to Google Drive at: {drive_path}")

        # Create directory in Drive if it doesn't exist
        os.makedirs(drive_path, exist_ok=True)

        # Copy files to Drive
        copied_count = 0
        for f_path in transcript_files:
            try:
                shutil.copy(f_path, os.path.join(drive_path, os.path.basename(f_path)))
                copied_count += 1
            except Exception as e:
                print(f"Failed to copy {f_path} to Drive: {e}")
        print(f"Successfully copied {copied_count} transcripts to Google Drive.")

    print(f"\nTranscription Summary:")
    print(f"Total files processed: {len(audio_files)}")
    print(f"Transcripts created: {len(transcript_files)}")

    return transcript_files


In [17]:
# Prompt the user for the API key manually
# gemini_api_key = input("Please enter your Gemini API Key: ")

In [14]:
# Example usage
input_directory = "soundcloud_2025-03-15_to_2025-03-16"

# Transcribe all audio files in the directory
transcripts = transcribe_audio_files(
    input_dir=input_directory,
    api_key=gemini_api_key, # Pass the manually entered API key
    language="so", # Somali language code
    save_to_drive=True
)

Error mounting Google Drive: Mountpoint must not already contain files
Continuing without Google Drive. Files will only be saved locally.
ffmpeg is already installed.
Using Gemini model: gemini-2.0-flash
Found 2 audio files to transcribe.
Formats to process: wav, mp3, m4a, flac, ogg
Skipping IDAACADDA 16-MAR-2025.mp3 - already transcribed

Transcribing: IDAACADDA 15-MAR-2025.mp3 (Format: mp3)
Successfully transcribed IDAACADDA 15-MAR-2025.mp3
Removed temporary conversion directory: soundcloud_2025-03-15_to_2025-03-16/_temp_gemini_conversion

Transcription Summary:
Total files processed: 2
Transcripts created: 2


In [15]:
# Example usage
file_path = "/content/gemini_transcripts_soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 15-MAR-2025.txt"
# file_path = "/content/gemini_transcripts_soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 16-MAR-2025.txt"

results = analyze_somali_transcript(file_path)

Analyzing file: IDAACADDA 15-MAR-2025.txt

==== Sample Lines ====
Line 1: Halkanina waa Radiyo Ergo ee codka arimaha bini'aadanimada oo fadhigiisu yahay
magalada Nairobi ee dalka kenya waxad naga dhageysanaysaan mowjado gaaban ee
dherer keedu yahay 22 ka mitir baan una dhiganta 25,670.00 maga hertz. Saacada
geeska afrika dhabari marka ay tahay sedexda ilaa afarta gelabnimo
Line 2: Waxaad sidoo kale naga dhagaysanaysaan qaara ka tirsan idaacadaha dalka iyo
barta aanu ku leenahay internetka ee fadhigeedu yahay radiyo ergo dot o w r g.
Line 3: Kulanti wanaag san eedageystayal waa sabti ay bisha maarso sanadku waa 2022 ku
soo dhawaada idaca Ergo ee maanta anigu ahsan madaale ayaa la socodsiinayaa
qodobada aadka macquul ka doontan waxa ka mid ah xooldhaqatada noolol xumagaa ku
waajahay iyo diganka cali waal ee gobal ku nado o markii daqalaah aan iyo biyo
yari ay u dhinten in ka badan xoolahoodi. Qoraal sokono wax magalada fasah dherr
e gobal ka baydh waa ugu dilah maareynta dhaarasho quusas

# 3. Performance Overview  

# 📄 Somali Audio Transcription Model Evaluation

This document outlines the performance of three different AI models used to transcribe Somali-language audio. The objective was to identify a model capable of accurately converting spoken Somali into text using the **correct Latin script**. The evaluation moved from general-purpose models to more specialized ones, culminating in the success of **Gemini 2.0 Flash**.

---

## 1. 🧠 OpenAI Whisper (Standard Model)

### 🔍 Summary:
OpenAI's standard Whisper model was the initial choice, given its general-purpose transcription capability.

- **✅ Language Detection:** Correctly identified Somali.
- **❌ Script Used:** Incorrectly transcribed text using the **Arabic script**, not the standard **Somali Latin script**.

### ⚠️ Key Issue:
The transcription output was nonsensical and unusable. It was filled with repeated Arabic phrases like:

> **"موضوع موضوع موضوع..."** (Arabic for "subject")

### 📉 Transcription Quality:
- Repetitive
- Unintelligible
- Wrong script

### 🧾 Conclusion:
**❌ Failure.** The model was unable to produce text in the appropriate script, making it ineffective for this task.

---

## 2. 🧬 Whisper Small Somali (`steja/whisper-small-somali`)

### 🔍 Summary:
A specialized model from Hugging Face, fine-tuned specifically for Somali, was tested to address the script issue.

- **✅ Script Used:** Correctly used **Somali Latin script**
- **✅ Language:** Identified and transcribed Somali

### ⚠️ Key Issue:
The model produced **hallucinatory repetitions**, frequently looping on phrases like:

> *"iyo iyo iyo," "dhul dhul dhul," "dheesho dheesho..."*

### 📉 Transcription Quality:
- Correct script, but
- High frequency of repetitive nonsense
- Difficult to interpret meaningful content

### 🧾 Conclusion:
**⚠️ Partial Success.** Script issue resolved, but the hallucinations rendered the transcriptions unreliable for practical use.

---

## 3. 🚀 Gemini 2.0 Flash (Google)

### 🔍 Summary:
The final model tested, **Gemini 2.0 Flash**, exhibited strong performance and significantly improved output quality.

- **✅ Script:** Somali Latin script
- **✅ Structure:** Coherent sentences and paragraphs
- **✅ Repetition:** Natural and minimal
- **✅ Accuracy:** High contextual relevance

### ✅ Transcription Quality:
- Well-structured
- Accurate
- Readable and ready for analysis

### 🧾 Conclusion:
**✅ Success.** Gemini 2.0 Flash delivered the best results. It provided **accurate**, **natural**, and **usable** transcriptions aligned with the project’s goals.

---


